# 03 — RAG multimodal : BGE-M3, CLIP et RRF

**Objectif :** retrouver les passages et les schémas pertinents pour une question.

Deux index sont construits :

- **texte** : BGE-M3, 1024 dimensions, multilingue ;
- **image** : CLIP ViT-B/32, 512 dimensions.

Les scores des deux modèles ne sont pas comparables. On fusionne donc les rangs
avec Reciprocal Rank Fusion (RRF), pas les scores.


## Étape 1 — Charger les artefacts

On charge les chunks et les images produits par le notebook 02. Aucun accès au PDF
n'est nécessaire.


In [1]:
import json
from maintidoc.config import PROCESSED_DIR

chunks = json.loads((PROCESSED_DIR / "chunks.json").read_text(encoding="utf-8"))
document = json.loads((PROCESSED_DIR / "document.json").read_text(encoding="utf-8"))

print("Chunks :", len(chunks))
print("Images :", len(document["images"]))

Chunks : 26
Images : 6


### Lecture du résultat

On doit retrouver 26 chunks et 6 images. Les métadonnées de page et d'identifiant
sont conservées.


## Étape 2 — Construire l'index texte BGE-M3

`TextIndex.build()` encode chaque passage avec BGE-M3 et normalise les vecteurs.
La similarité cosinus devient un simple produit scalaire.

L'index est sauvegardé dans `data/processed/index/`.


In [2]:
from maintidoc.retrieval import TextIndex

text_index = TextIndex.build(chunks)
print("Passages indexés :", len(text_index.chunks))
print("Dimension :", text_index.embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Passages indexés : 26
Dimension : (26, 1024)


### Lecture du résultat

La dimension doit être `(26, 1024)`. Le premier chargement de BGE-M3 peut prendre
plusieurs minutes ; les suivants utilisent le cache local.


## Étape 3 — Rechercher un passage

On interroge l'index avec la question. `search()` renvoie les passages classés par
similarité cosinus, avec page et identifiant source.


In [3]:
question = "Que signifie le voyant orange AL-01 ?"

hits = text_index.search(question, top_k=5)
for hit in hits:
    print(f"#{hit['rank']} page {hit['page']} | {hit['element_id']} | {hit['score']:.3f}")
    print("  ", hit["text"][:160].replace("\n", " "))

#1 page 5 | P05-TEXT | 0.614
   SAN-MAN-001 | MaintiDoc | Manuel pedagogique fictif Page 5 4. Voyants, evenements et interpretation FIG-04 : signification uniquement conventionnelle dans le si
#2 page 5 | P05-TAB01 | 0.583
   | Code | Etat | Signification pour le simulateur | Reponse documentaire | |---|---|---|---| | AL-01 | Orange | Chemin FC degrade | Verifier DIAG-FC-01 et FIG-02
#3 page 7 | P07-TAB02 | 0.549
   | ID | Question | Source attendue | |---|---|---| | Q01 | Quel est le role des deux fabrics ? | FIG-01, FIG-02 | | Q02 | Ou intervient le zoning ? | FIG-02 | | 
#4 page 7 | P07-TEXT | 0.508
   ALARM-TAB-01, FIG-02, INC-104 Statut Brouillon - approbation requise 7. Jeu de questions pour le RAG multimodal ID Question Source attendue Q01 Quel est le role
#5 page 5 | P05-TEXT | 0.497
   preventive imposee Interpretation d une photographie Exemple de question : « Le voyant orange est allume : quelle est la cause ? » Une photographie du voyant ne


### Lecture du résultat

La page 5 doit dominer : elle contient le tableau des alarmes et le texte sur les
voyants. Les scores sont des similarités, pas des probabilités.


## Étape 4 — Construire l'index image CLIP

CLIP encode les 5 schémas. La couverture est exclue. La requête visuelle est
écrite en anglais, car CLIP ViT-B/32 est surtout anglophone.


In [4]:
from maintidoc.retrieval import ImageIndex

image_index = ImageIndex.build(document["images"])
print("Schémas indexés :", len(image_index.images))
print("Dimension :", image_index.embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Schémas indexés : 5
Dimension : (5, 512)


### Lecture du résultat

La dimension doit être `(5, 512)`. Les 5 figures sont indexées.


## Étape 5 — Rechercher un schéma

On compare une requête visuelle anglaise aux embeddings des figures. Le score est
une similarité, pas une preuve que le schéma répond à la question.


In [5]:
visual_query = "orange warning LED storage array front panel"

for hit in image_index.search(visual_query, top_k=3):
    print(f"#{hit['rank']} {hit['figure_id']} page {hit['page']} | {hit['score']:.3f}")

#1 FIG-02 page 3 | 0.200
#2 FIG-04 page 5 | 0.194
#3 FIG-01 page 2 | 0.159


### Lecture du résultat

CLIP peut classer FIG-04 (voyants) ou FIG-02 (Fibre Channel) selon la formulation.
C'est une limite connue : la similarité visuelle n'est pas une compréhension du
schéma. La fusion avec le texte compense partiellement.


## Étape 6 — Fusion RRF

On fusionne les rangs au niveau des pages. La formule est :

$$\mathrm{RRF}(d)=\sum_{r}\frac{1}{k+\mathrm{rank}_r(d)}$$

Pourquoi les rangs ? Parce qu'un score BGE-M3 et un score CLIP ne vivent pas sur
la même échelle.


In [6]:
from maintidoc.retrieval import rrf_fuse

pages = rrf_fuse(hits, image_index.search(visual_query, top_k=5))
for page in pages:
    print(page)

{'page': 5, 'rrf': 0.03252247488101534, 'sources': ['texte', 'visuel']}
{'page': 3, 'rrf': 0.01639344262295082, 'sources': ['visuel']}
{'page': 7, 'rrf': 0.015873015873015872, 'sources': ['texte']}
{'page': 2, 'rrf': 0.015873015873015872, 'sources': ['visuel']}


### Lecture du résultat

La page 5 doit remonter en tête, avec les sources `texte` et `visuel`. Les pages
3 et 2 peuvent apparaître pour les schémas.


## Étape 7 — Construire le dossier de preuves

On garde quelques passages, un tableau et au plus une image. Le contexte est borné
pour maîtriser la mémoire et la latence.

Chaque preuve garde un identifiant et une page : c'est la base des citations.


In [7]:
from maintidoc.retrieval import build_evidence

evidence, images = build_evidence(hits, image_index.search(visual_query, top_k=5), pages)
for item in evidence:
    print(item["id"], "| page", item["page"], "|", item["type"])
print("Images :", [im["figure_id"] for im in images])

P05-TEXT | page 5 | text
P05-TAB01 | page 5 | table
P07-TAB02 | page 7 | table
P07-TEXT | page 7 | text
P05-TEXT | page 5 | text
P05-IMG01 | page 5 | image
Images : ['FIG-04']


### Lecture du résultat

Les preuves sont peu nombreuses et toutes identifiables. L'image choisie doit
correspondre à la page la mieux classée, pas seulement au meilleur score CLIP.


## Étape 8 — Évaluation de la recherche

On exécute le script d'évaluation sur 14 questions de référence. Les métriques
sont :

- `Recall@5` pour les pages ;
- `Recall@3` pour les figures ;
- `MRR` pour le premier rang pertinent.


In [8]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "eval/run_eval.py"],
    cwd=str(PROCESSED_DIR.parents[1]),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Questions : 14
recall_text@5: 0.857
recall_rrf: 0.929
recall_image@3: 0.714
mrr_page: 0.667
Résultats : D:\EMSI-RelatedGeneralFiles-S9\IA generative Visuelle + Textuelle\maintidoc\eval\results.csv



### Lecture du résultat

Les scores obtenus sur ce corpus sont :

- `Recall@5` texte : **0,857** ;
- `Recall@5` après RRF : **0,929** ;
- `Recall@3` figures : **0,714** ;
- `MRR` page : **0,667**.

Ce ne sont pas des scores de production : le corpus est petit et les questions
sont proches du manuel.


## Bilan

- BGE-M3 indexe 26 passages en 1024 dimensions.
- CLIP indexe 5 schémas en 512 dimensions.
- RRF améliore le rappel pages.
- Le dossier de preuves est borné et traçable.

**Étape suivante :** envoyer les preuves au modèle et vérifier les citations.
